# Build the map's data

GEOG 464 · A2 · Montreal walking-distance map

Run the cells from top to bottom. The last code cell is the check cell: it reads
the four GeoJSON files back from disk and tests their counts, the values
attached to them, and the parks' areas. Nothing here needs a package outside
the standard library.

In [2]:
from pathlib import Path
import json
import sys


def find_project(start):
    for folder in [start, *start.parents]:
        if (folder / 'site' / 'build.py').exists():
            return folder
    raise SystemExit('Run this notebook from inside the project folder.')


PROJECT = find_project(Path.cwd())
SITE = PROJECT / 'site'
sys.path.insert(0, str(SITE))

import build

print('project:', PROJECT)
print('site:   ', SITE)

project: c:\Users\arthur\Documents\GEOG464\project
site:    c:\Users\arthur\Documents\GEOG464\project\site


## What the sources hold

The same three files step 2 reads, read here first, with the same functions the
build uses. The counts below are what each reader hands back: the metro rows,
split into stations, entrances and the platform rows that are read only for
their station names; the city's street lines, its troncons, one line for each
stretch of street and already with the motorways dropped; and the green
spaces, exactly as the city drew them.

The street file is read once per run and the answer kept, so this cell and the
step 3 cell below read the same lines in the same order rather than each
fetching a copy of its own.

In [3]:
metro_rows, platform_rows = build.read_metro_rows()
parks_source = build.read_parks()
street_lines = build.read_streets()

print('metro rows:', len(metro_rows), '+', len(platform_rows), 'platform rows')
print('green spaces:', len(parks_source['features']))
print('street troncons:', len(street_lines))

del metro_rows, platform_rows, parks_source, street_lines

8880 rows in stops.txt, 274 are metro rows: 202 stations and entrances written, 72 platform rows read for their names and URLs
2245 features in espace_vert.json
45904 troncons read and projected to metres, 277987 points, 5411 km of street
2127 troncons dropped as motorway or motorway service, 484 km not walked
metro rows: 202 + 72 platform rows
green spaces: 2245
street troncons: 45904


## Step 2 - the three base files

Reads the three sources it maps, drops the rows it does not map (each removal is
printed), and writes `site/data/metro_stops.geojson`,
`site/data/city_parks.geojson` and `site/data/intersections.geojson`.


In [4]:
build.build_geojson()

8880 rows in stops.txt, 274 are metro rows: 202 stations and entrances written, 72 platform rows read for their names and URLs
202 features built, 0 rows skipped
68 stations, popup fields added
Popup preview:
  Metro Angrignon (Station) | https://www.stm.info/fr/infos/reseaux/metro/angrignon
  Metro Angrignon (Entrance) | Wheelchair accessible
All checks passed. 202 features.
Written to C:\Users\arthur\Documents\GEOG464\project\site\data\metro_stops.geojson (139 KB)
2245 features in espace_vert.json
1981 features built, 264 street spaces skipped
1981 green spaces, popup fields added
1981 green spaces, biggest first
Popup preview:
  [city] Grand parc de l'Ouest | Type: City Park | Size: 295.632 ha
  [city] Parc le Boisé Riverain | Type: Borough Park | Size: 3.496 ha
  [derived] Royal Montréal Golf course | Type: Other green space | Size: 272.308 ha
  [derived] Centre Sportif George Springate (Park) | Type: Borough Park | Size: 15.946 ha
  [derived] Lieu Historique National du Canada du-

## Step 3 - the 22 walk layers

A layer is one question the page can ask - which kind of green space, of which
size, or which kind of metro entrance - and its answer is one number for every
street corner on the map.

First the corners are put back on the network of streets the walks run over.
Then fourteen of the layers are measured directly: one walk search per layer,
spreading out from every place that layer is about. The other eight are the
minimum of a few of those fourteen, so a corner's answer for "any size" is the
best of its four sizes. Five checks then look at the answers from five
different directions.

The answers are written onto the corners in
`site/data/intersections.geojson`, and the 34 outlines the page's mask is cut
round go to `site/data/boroughs.geojson`. This is the slow cell.

In [5]:
build.build_walk_times()

24669 street corners read from the intersections file: 3 ends 16242, 4 ends 8156, 5 ends 257, 6 ends 13, 7 ends 1
45904 troncons read and projected to metres, 277987 points, 5411 km of street
2127 troncons dropped as motorway or motorway service, 484 km not walked
134 metro entrances at 68 stations, 28 of them step-free
1981 green spaces read: arrondissement 1384, autre 184, grand_parc 139, jardin 1, pending 270, unknown 3
  by size: 0to1 1394, 1to5 411, 5to10 71, 10up 105
34 published outlines read, 19 of them the page draws, from LaSalle to Westmount
19360 of the 24669 corners are in the 19 boroughs the page draws: LaSalle 890, Villeray-Saint-Michel-Parc-Extension 1097, Rosemont-La Petite-Patrie 993, Le Plateau-Mont-Royal 749, Côte-des-Neiges-Notre-Dame-de-Grâce 1123, L'Île-Bizard-Sainte-Geneviève 508, Anjou 686, Verdun 520, Le Sud-Ouest 937, Mercier-Hochelaga-Maisonneuve 1616, Lachine 604, Saint-Léonard 915, Montréal-Nord 907, Outremont 229, Pierrefonds-Roxboro 1228, Rivière-des-Pra

## Check cell

This reads the files that were just written, not the objects still in memory,
so it tests the GeoJSON the page loads. It checks the counts, the values the
page reads, the shape of every walk, and every park's area against the city's
own `SUPERFICIE` column - with the area worked out here from the park's own
shape, in metres, rather than taken from the file.

In [6]:
DATA = SITE / 'data'
metro = json.loads((DATA / 'metro_stops.geojson').read_text(encoding='utf-8'))
parks = json.loads((DATA / 'city_parks.geojson').read_text(encoding='utf-8'))
dots = json.loads((DATA / 'intersections.geojson').read_text(encoding='utf-8'))
boroughs = json.loads((DATA / 'boroughs.geojson').read_text(encoding='utf-8'))

# Counts this build pins in its own checks.
assert len(metro['features']) == 202
assert sum(1 for f in metro['features']
           if f['properties']['location_type'] == '1') == 68
assert sum(1 for f in metro['features']
           if f['properties']['location_type'] == '2') == 134
assert len(parks['features']) == 1981
assert len(dots['features']) == 24669
assert len(dots['walk_layers']) == 22
assert len(boroughs['features']) == 34
assert sum(1 for f in boroughs['features']
           if f['properties']['drawn']) == 19

# The properties the page reads, and every ring closed.
metro_keys = {'popup_name', 'popup_type', 'popup_accessible', 'popup_url',
              'location_type'}
park_keys = {'popup_title', 'popup_title_source', 'popup_kind',
             'popup_kind_key', 'popup_size', 'SUPERFICIE'}
for feature in metro['features']:
    assert metro_keys <= set(feature['properties']), feature['properties']
for feature in parks['features']:
    props = feature['properties']
    assert park_keys <= set(props), sorted(props)
    geometry = feature['geometry']
    polygons = ([geometry['coordinates']] if geometry['type'] == 'Polygon'
                else geometry['coordinates'])
    for polygon in polygons:
        for ring in polygon:
            assert ring[0] == ring[-1], props['OBJECTID']

# Each corner carries one [seconds, destination] pair per layer.
for feature in dots['features']:
    walk = feature['properties']['walk']
    assert len(walk) == len(dots['walk_layers']), feature['properties']['id']
    for pair in walk:
        assert isinstance(pair, list) and len(pair) == 2
        assert all(isinstance(value, int) for value in pair)


# Area: shoelace over every ring, projected to metres, against SUPERFICIE.
def ring_area(ring):
    points = [build.to_metres(lon, lat) for lon, lat in ring]
    twice = sum(x0 * y1 - x1 * y0
                for (x0, y0), (x1, y1) in zip(points, points[1:]))
    return abs(twice) / 2.0


def shape_area(geometry):
    polygons = ([geometry['coordinates']] if geometry['type'] == 'Polygon'
                else geometry['coordinates'])
    return sum(ring_area(polygon[0])
               - sum(ring_area(hole) for hole in polygon[1:])
               for polygon in polygons)


worst = 0.0
worst_ratio = 1.0
for feature in parks['features']:
    city_ha = float(feature['properties']['SUPERFICIE'])
    measured_ha = shape_area(feature['geometry']) / 10000.0
    difference = abs(measured_ha - city_ha)
    assert difference <= max(0.001, 0.005 * city_ha), (
        feature['properties']['OBJECTID'], measured_ha, city_ha)
    worst = max(worst, difference)
    worst_ratio = max(worst_ratio, measured_ha / city_ha)

print('check passed: 202 metro points, 1,981 parks, 24,669 corners, '
      '22 walk layers, 34 outlines')
print(f'area: every park matches SUPERFICIE within 0.5%; '
      f'worst {worst:.4f} ha (ratio {worst_ratio:.5f})')

check passed: 202 metro points, 1,981 parks, 24,669 corners, 22 walk layers, 34 outlines
area: every park matches SUPERFICIE within 0.5%; worst 0.0071 ha (ratio 1.00010)


## What this proves

The four files the page loads are the files this notebook wrote, and their
counts, their values and the shape of every walk agree with the sources. A
GeoJSON feature keeps its values beside its shape, under the name `properties`,
and that is what is being read here.

What this does not prove is that the walks are right in the sense a routing
service would: the build's own five checks in step 3, and `node site/app.js`,
cover that. What this cell adds is the independent area test - every park's
area worked out from its own shape and held against the city's own size column
- and a read-back of the files from disk.